# Notebook 4: Out-of-Sample Backtest

Applies the parameters fixed on training data in Notebook 3 to the held-out test
set, and benchmarks the HYSYS-weighted spread against the conventional 3:2:1
crack spread under identical trading mechanics.

Nothing is re-fitted here. The entry threshold, exit threshold, lookback window,
holding period and transaction cost all come from `../data/ou_parameters.json`,
chosen before the test set was touched.

## Look-ahead bias

PnL uses `signal.shift(1)`: a position established from day *t−1*'s close earns
day *t*'s move. Trades are entered the day after the close that generated the
signal, so the strategy never acts on information unavailable at the decision
point. The rolling z-score window is likewise strictly backward-looking.

## Benchmark

The 3:2:1 spread is traded with the same z-score lookback, the same entry and
exit thresholds, the same minimum holding period and the same costs. This
isolates the effect of the HYSYS-derived weighting itself rather than differences
in trading logic.

## Interpreting the result

Trade count is reported alongside every Sharpe figure. A Sharpe computed from a
handful of trades carries no statistical content, and the notebook flags this
explicitly rather than reporting the number as though it were a finding.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from signal_generator import generate_signals, apply_holding_period, compute_pnl
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# LOAD - all parameters were fixed on TRAIN data in Notebook 3
# ============================================================

df = pd.read_csv('../data/spread_full_with_split.csv', index_col=0, parse_dates=True)

with open('../data/ou_parameters.json') as f:
    p = json.load(f)

ENTRY  = p['entry_threshold']
EXIT   = p['exit_threshold']
LOOK   = p['lookback']
HOLD   = p['min_holding_days']
COST   = p['transaction_cost']
SPLIT  = pd.Timestamp(p['split_date'])

print('Parameters fixed on TRAIN data, applied unchanged here:')
for k in ['entry_threshold', 'exit_threshold', 'lookback',
          'min_holding_days', 'transaction_cost']:
    print(f'  {k:18s} {p[k]!r:>8}  ({type(p[k]).__name__})')
print(f'  split_date         {p["split_date"]}')
print(f'\nOU half-life from train fit: {p["half_life"]:.1f} days')

In [ ]:
# ============================================================
# Z-SCORES - rolling, strictly backward-looking
# ============================================================

for col, tag in [('spread_hysys', 'hysys'), ('crack_321', '321')]:
    m = df[col].rolling(LOOK).mean()
    s = df[col].rolling(LOOK).std()
    df[f'z_{tag}'] = (df[col] - m) / s

df = df.dropna(subset=['z_hysys', 'z_321'])

train_mask = df.index <  SPLIT
test_mask  = df.index >= SPLIT

print(f'After rolling warm-up:')
print(f'  Train: {train_mask.sum()} obs')
print(f'  Test:  {test_mask.sum()} obs')

In [ ]:
# ============================================================
# SIGNALS + PnL - identical mechanics for both strategies
# ============================================================

strategies = {
    'hysys': ('spread_hysys', 'HYSYS partial spread'),
    '321':   ('crack_321',    'Generic 3:2:1 benchmark')
}

for tag, (col, label) in strategies.items():
    sig = apply_holding_period(
              generate_signals(df[f'z_{tag}'], ENTRY, EXIT), HOLD)
    bt  = compute_pnl(sig, df[col], COST)
    df[f'pnl_{tag}']    = bt['pnl_net']
    df[f'trade_{tag}']  = bt['trade']
    df[f'equity_{tag}'] = bt['pnl_net'].cumsum()

print('Signals and PnL computed for both strategies.')

In [ ]:
# ============================================================
# RESULTS - train vs out-of-sample, both strategies
# ============================================================

def metrics(pnl, trades, n_days):
    eq = pnl.cumsum()
    sd = pnl.std()
    return {
        'Total PnL ($/bbl)': round(pnl.sum(), 2),
        'Sharpe':            round((pnl.mean() / sd) * np.sqrt(252), 2) if sd > 0 else 0.0,
        'Ann. vol ($/bbl)':  round(sd * np.sqrt(252), 2),
        'Max DD ($/bbl)':    round((eq - eq.cummax()).min(), 2),
        'Win rate':          f'{(pnl > 0).mean():.1%}',
        'Trades':            int(trades.sum() / 2),
        'Turnover (/day)':   round(trades.sum() / n_days, 4)
    }

rows = []
for period, mask in [('Train (in-sample)', train_mask),
                     ('Test (OUT-OF-SAMPLE)', test_mask)]:
    for tag, (col, label) in strategies.items():
        rows.append({
            'Period': period, 'Strategy': label,
            **metrics(df.loc[mask, f'pnl_{tag}'],
                      df.loc[mask, f'trade_{tag}'], mask.sum())
        })

table = pd.DataFrame(rows)
print('=' * 100)
print('PERFORMANCE - train vs out-of-sample, HYSYS vs 3:2:1 benchmark')
print('=' * 100)
print(table.to_string(index=False))

oos = table[table['Period'] == 'Test (OUT-OF-SAMPLE)']
h = oos[oos['Strategy'].str.contains('HYSYS')].iloc[0]
b = oos[oos['Strategy'].str.contains('3:2:1')].iloc[0]

print('\nOUT-OF-SAMPLE HEADLINE')
print(f'  HYSYS:      Sharpe {h["Sharpe"]:>6}   {h["Trades"]:>3} trades   PnL ${h["Total PnL ($/bbl)"]}/bbl')
print(f'  Benchmark:  Sharpe {b["Sharpe"]:>6}   {b["Trades"]:>3} trades   PnL ${b["Total PnL ($/bbl)"]}/bbl')

if h['Trades'] < 30:
    print(f'\n  *** {h["Trades"]} out-of-sample trades is too few for a reliable Sharpe.   ***')
    print('  *** Report the strategy qualitatively rather than quoting the number. ***')

In [ ]:
# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(df.index, df['equity_hysys'], color='#2c3e50', linewidth=1.4,
             label='HYSYS partial spread')
axes[0].plot(df.index, df['equity_321'], color='#7f8c8d', linewidth=1.1,
             linestyle='--', label='3:2:1 benchmark')
axes[0].axvline(SPLIT, color='red', linestyle=':', linewidth=1.5, label='Train/test split')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('Cumulative PnL ($/bbl)')
axes[0].set_title('Equity curves, net of transaction costs')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(df.index, df['z_hysys'], color='#7f8c8d', linewidth=0.6)
for lvl, c in [(0, 'black'), (ENTRY, '#e74c3c'), (-ENTRY, '#27ae60'),
               (EXIT, '#3498db'), (-EXIT, '#3498db')]:
    axes[1].axhline(lvl, color=c, linewidth=0.9, linestyle='--', alpha=0.7)
axes[1].axvline(SPLIT, color='red', linestyle=':', linewidth=1.5)
axes[1].set_ylabel('Z-score (HYSYS)')
axes[1].set_title(f'Signal: entry +/-{ENTRY} sigma, exit +/-{EXIT} sigma')
axes[1].grid(alpha=0.3)

dd_h = df['equity_hysys'] - df['equity_hysys'].cummax()
dd_b = df['equity_321']   - df['equity_321'].cummax()
axes[2].fill_between(df.index, dd_h, 0, alpha=0.4, color='#2c3e50', label='HYSYS')
axes[2].plot(df.index, dd_b, color='#7f8c8d', linewidth=1, linestyle='--', label='3:2:1')
axes[2].axvline(SPLIT, color='red', linestyle=':', linewidth=1.5)
axes[2].set_ylabel('Drawdown ($/bbl)')
axes[2].set_xlabel('Date')
axes[2].set_title('Drawdown')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SAVE
# ============================================================

table.to_csv('../data/performance_summary.csv', index=False)
df.to_csv('../data/backtest_results_full.csv')

print('Saved performance_summary.csv and backtest_results_full.csv')
print()
print('Markdown table for the README:')
print()
print(table.to_markdown(index=False))